# Can We Predict Which Orders Will Be Refunded?

**Request (Operations / Inventory):** Refunded items come back into stock. Can we tell, at the time of purchase, which orders are most likely to be refunded, so returned stock can be planned for?

**Data:** `fct_orders` from the Vantra dbt pipeline (BigQuery). Revenue rule applied: $0 and missing-price orders excluded. Only orders where a refund could still be recorded (see Part A).

**Approach:** Explore refund rates by order trait → decide features and training window → train a simple model → check whether its riskiest orders really refund more often than average.

## 1. Setup
Connect Colab to the BigQuery project so the notebook queries the same tested dbt marts used across the project.

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Authenticated")

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="vantra-514719")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

## Part A — Explore
### 2. When do refunds stop being recorded?
If refunds stopped being recorded at some point, every later order looks "not refunded" even if it was. Training on those orders would teach the model the wrong answer.

In [ ]:
sql = """
select max(refund_date) as last_refund_date, max(purchase_date) as last_purchase_date
from dbt_bpan.fct_orders;
"""

client.query(sql).to_dataframe()

The last recorded refund is **Aug 7, 2021**, but orders continue into 2022. Orders after that date can't be trusted as "not refunded."

**Training window:** orders purchased on or before **Jul 14, 2021**, about three weeks before the last recorded refund, so orders that were refunded shortly after purchase still had time to be recorded.

### 3. Pull the orders
One row per order in the training window, with the revenue rule applied.

In [ ]:
sql = """
select * from dbt_bpan.fct_orders
where purchase_date <= '2021-07-14'
  and region is not null
  and is_zero_price is false
  and is_missing_price is false;
"""

orders_df = client.query(sql).to_dataframe()
orders_df.head()

### 4. Refund rate by order trait
Overall refund rate first, then the refund rate within each category. A category well above the overall rate is a possible risk driver.

In [ ]:
print("Overall refund rate:", round(orders_df['is_refunded'].mean(), 3), "\n")

orders_df['purchase_year'] = pd.to_datetime(orders_df['purchase_date']).dt.year

traits = ['product_name','brand','marketing_channel','purchase_platform','region','is_loyalty_member','purchase_year']

for trait in traits:
  refund_rate = orders_df.groupby(trait)['is_refunded'].mean().sort_values(ascending=False)
  print(refund_rate.round(3), "\n")

## Decide
- **Features:** product, marketing channel, purchase platform, region and loyalty membership. These are all known at the moment of purchase.
- **Dropped:** purchase year, because a model used on future orders can't learn anything from a year it has never seen. Brand, because it is already contained in the product (each product has one brand).
- **Training window:** orders on or before Jul 14, 2021 (see section 2).
- **What "good" means:** for inventory planning, flags need to be trustworthy. A false alarm means planning for returned stock that never arrives. So the measure is **precision**: of the orders we flag, how many are actually refunded?

## Part B — Build
### 5. Train the model
- Text categories become 0/1 columns, since the model only reads numbers. One category per feature is dropped and becomes the baseline the others are compared against.
- 25% of orders are held back as a test set the model never sees during training. `stratify` keeps the refund rate the same in both sets.
- A **logistic regression** learns a weight for each trait, then gives every order a refund probability.

In [ ]:
features = ['product_name','marketing_channel','purchase_platform','region','is_loyalty_member']

X = pd.get_dummies(orders_df[features], dtype='int64', drop_first=True)
y = orders_df['is_refunded'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=29, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

## Part C — Test
### 6. Why accuracy is misleading here
About 92% of orders are not refunded, so a model that never flags anything already scores 92% accuracy. No order reaches a 50% refund probability, so the default yes/no prediction flags nothing. Instead, we use each order's refund **probability** and look at the riskiest orders.

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]   # refund probability for each test order

print("Test orders:", len(y_test))
print("Lowest / highest predicted probability:", round(y_prob.min(), 3), "/", round(y_prob.max(), 3))

### 7. Do the riskiest orders actually refund more?
- **Baseline:** the refund rate across all test orders. Flagging orders at random would hit this rate.
- **Top 5%:** sort test orders by predicted probability and take the riskiest 5%. Their actual refund rate is the model's precision.

In [ ]:
scored_df = pd.DataFrame({"is_refunded": y_test.values, "refund_prob": y_prob}).sort_values(
    "refund_prob", ascending=False
)

top_n = int(len(scored_df) * 0.05)

baseline_rate = scored_df['is_refunded'].mean()
top_rate = scored_df['is_refunded'].head(top_n).mean()

print("Baseline refund rate (all test orders):", round(baseline_rate, 3))
print(f"Refund rate in top 5% ({top_n} orders):", round(top_rate, 3))
print("Lift:", round(top_rate / baseline_rate, 1), "x")

### 8. What drives refund risk?
Each coefficient shows how a trait moves refund risk compared with the dropped baseline category (27in 4K Gaming Monitor, affiliate channel, mobile app, APAC, non-member). Positive raises risk; negative lowers it. Features were scaled, so read the coefficients as a ranking rather than exact effect sizes.

In [ ]:
drivers_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

drivers_df

## 9. Verdict for Operations
The model finds **real but moderate signal**. The riskiest 5% of orders refund at **15.1%**, about **1.9×** the **7.8%** average.

**What drives refund risk:**
- **Loyalty members** are the strongest driver, consistent with the loyalty test, where members also bought again less often.
- **Laptops** (MacBook Air, ThinkPad) are the riskiest products. Higher-priced items come back more often.
- **Website orders** are riskier than app orders.
- **Cheap accessories** are the safest, especially the charging cable pack.
- Region and marketing channel barely matter.

**Recommendation:** use the model to estimate returned stock in aggregate, especially laptops, not to act on individual orders. 85% of flagged orders are still not refunded.

**Caveats:**
- Refunds stop being recorded after Aug 2021, so the model learned only from orders through mid-2021.
- These are associations, not proof that a trait *causes* refunds.
- With only five order-level traits, precision is limited. Order-level details like return reason or delivery time would likely improve it.